# Customer Churn Analysis
#### An insight on Telco customer churn in California
_ETL process in Python_

Before loading the data in Fabric, the main part of the ETL process is performed in Python. 
The steps that will be followed are the following:

1. Quick exploratory analysis after loading the source data in a Bronze Layer
2. Creation of a Silver Layer and a transaction table derived from the source data, tracking month by month customer tenure and capturing dynamic changes and customer churn across time dimension
3. Quick data pre-processing (handling missing data and outliers)
4. Loading clean and transformed data into the Gold Layer

Afterwards the analysis will continue in Fabric and Power BI.

### 1. Exploratory Analysis
#### Bronze Layer

In [7]:
import pandas as pd
import numpy as np

file_path = '/lakehouse/default/Files/TelcoCustomerChurn.csv'
df = pd.read_csv(file_path)

if 'Quarter' in df.columns:
    df.drop(columns=['Quarter'], inplace=True)

np.random.seed(42)
start_date = pd.to_datetime('2025-01-01')
end_date = pd.to_datetime('2025-12-31')
days_range = (end_date - start_date).days
random_days = np.random.randint(0, days_range + 1, size=len(df))

df['Date'] = start_date + pd.to_timedelta(random_days, unit='D')
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# Save directly over the original file
df.to_csv(file_path, index=False)
print("Original file updated successfully with 'Date' column!")

StatementMeta(, ada6040f-390e-46ee-9fc0-f83e99d6aeb7, 12, Finished, Available, Finished, False)

Original file updated successfully with 'Date' column!


In [ ]:
pd.set_option('display.max_columns', None)
df.head(10)

StatementMeta(, ada6040f-390e-46ee-9fc0-f83e99d6aeb7, 6, Finished, Available, Finished, False)

,CustomerID,Gender,Age,Under30,SeniorCitizen,Married,Dependents,NumberofDependents,Country,State,City,ZipCode,Latitude,Longitude,Population,ReferredaFriend,Number_of_Referrals,TenureinMonths,Offer,PhoneService,AvgMonthlyLongDistanceCharges,MultipleLines,InternetService,InternetType,AvgMonthlyGBDownload,OnlineSecurity,OnlineBackup,DeviceProtectionPlan,PremiumTechSupport,StreamingTV,StreamingMovies,StreamingMusic,UnlimitedData,Contract,PaperlessBilling,PaymentMethod,MonthlyCharge,TotalCharges,TotalRefunds,TotalExtraDataCharges,TotalLongDistanceCharges,TotalRevenue,SatisfactionScore,CustomerStatus,ChurnLabel,ChurnScore,CLTV,ChurnCategory,ChurnReason,Date
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,Los Angeles,90022,34.023810,-118.156582,68701,No,0,1,NaN,No,0.00,No,Yes,DSL,8,No,No,Yes,No,No,Yes,No,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data,2025-04-13
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,Los Angeles,90063,34.044271,-118.185237,55668,Yes,1,8,Offer E,Yes,48.85,Yes,Yes,Fiber Optic,17,No,Yes,No,No,No,No,No,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer,2025-12-15
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,Los Angeles,90065,34.108833,-118.229715,47534,No,0,18,Offer D,Yes,11.33,Yes,Yes,Fiber Optic,52,No,No,No,No,Yes,Yes,Yes,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer,2025-09-28
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,Inglewood,90303,33.936291,-118.332639,27778,Yes,1,25,Offer C,Yes,19.76,No,Yes,Fiber Optic,12,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services,2025-04-17
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,Whittier,90602,33.972119,-118.020188,26265,Yes,1,37,Offer C,Yes,6.33,Yes,Yes,Fiber Optic,14,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges,2025-03-13
5,4412-YLTKF,Female,72,No,Yes,No,Yes,1,United States,California,Pico Rivera,90660,33.989524,-118.089299,63288,No,0,27,Offer C,Yes,3.33,Yes,Yes,Fiber Optic,18,No,No,Yes,No,No,No,No,No,Month-to-Month,Yes,Bank Withdrawal,78.05,2135.50,0.00,10,89.91,2235.41,1,Churned,Yes,95,4638,Competitor,Competitor had better devices,2025-07-08
6,0390-DCFDQ,Female,76,No,Yes,Yes,Yes,2,United States,California,Los Alamitos,90720,33.794990,-118.065591,21343,Yes,1,1,Offer E,Yes,15.28,No,Yes,Fiber Optic,30,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Mailed Check,70.45,70.45,0.00,0,15.28,85.73,2,Churned,Yes,76,3964,Other,Don't know,2025-01-21
7,3445-HXXGF,Male,66,No,Yes,Yes,No,0,United States,California,Sierra Madre,91024,34.168686,-118.057505,10558,Yes,6,58,Offer B,No,0.00,No,Yes,DSL,24,No,Yes,Yes,No,No,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,45.30,2651.20,40.95,0,0.00,2610.25,1,Churned,Yes,91,5444,Dissatisfaction,Service dissatisfaction,2025-04-13
8,2656-FMOKZ,Female,70,No,Yes,No,Yes,2,United States,California,Pasadena,91106,34.139402,-118.128658,23742,No,0,15,Offer D,Yes,44.07,Yes,Yes,Fiber Optic,19,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Mailed Check,74.45,1145.70,0.00,0,661.05,1806.75,2,Churned,Yes,91,5717,Dissatisfaction,Limited range of services,2025-05-02
9,2070-FNEXE,Female,77,No,Yes,No,Yes,2,United States,California,Pasadena,91107,34.159007,-118.087353,32369,No,0,7,Offer E,Yes,26.95,No,Yes,Fiber Optic,18,Yes,No,No,No,No,No,No,No,Month-to-Month,No,Bank Withdrawal,76.45,503.60,11.05,0,188.65,681.20,2,Churned,Yes,81,4419,Price,Lack of affordable download/upload speed,2025-08-03


In [ ]:
# Numeric variables 

df.describe().T

StatementMeta(, ada6040f-390e-46ee-9fc0-f83e99d6aeb7, 8, Finished, Available, Finished, False)

,count,mean,std,min,25%,50%,75%,max
Age,7043.0,46.509726,16.750352,19.000000,32.000000,46.000000,60.000000,80.000000
NumberofDependents,7043.0,0.468692,0.962802,0.000000,0.000000,0.000000,0.000000,9.000000
ZipCode,7043.0,93486.070567,1856.767505,90001.000000,92101.000000,93518.000000,95329.000000,96150.000000
Latitude,7043.0,36.197455,2.468929,32.555828,33.990646,36.205465,38.161321,41.962127
Longitude,7043.0,-119.756684,2.154425,-124.301372,-121.788090,-119.595293,-117.969795,-114.192901
Population,7043.0,22139.603294,21152.392837,11.000000,2344.000000,17554.000000,36125.000000,105285.000000
Number_of_Referrals,7043.0,1.951867,3.001199,0.000000,0.000000,0.000000,3.000000,11.000000
TenureinMonths,7043.0,32.386767,24.542061,1.000000,9.000000,29.000000,55.000000,72.000000
AvgMonthlyLongDistanceCharges,7043.0,22.958954,15.448113,0.000000,9.210000,22.890000,36.395000,49.990000
AvgMonthlyGBDownload,7043.0,20.515405,20.418940,0.000000,3.000000,17.000000,27.000000,85.000000


In [ ]:
# Categorical variables

df.describe(include = ['O']).T

StatementMeta(, ada6040f-390e-46ee-9fc0-f83e99d6aeb7, 9, Finished, Available, Finished, False)

,count,unique,top,freq
CustomerID,7043,7043,8779-QRDMV,1
Gender,7043,2,Male,3555
Under30,7043,2,No,5642
SeniorCitizen,7043,2,No,5901
Married,7043,2,No,3641
Dependents,7043,2,No,5416
Country,7043,1,United States,7043
State,7043,1,California,7043
City,7043,1106,Los Angeles,293
ReferredaFriend,7043,2,No,3821
